# **PART 1**

In [1]:
import time
import multiprocessing
import cProfile

# **Part 2**

In [12]:
def compute_sum(data):
  total = 0
  for x in data:
    total += x * x
  return total

In [11]:
data = list(range(1, 10_000_000))

In [4]:
start_time = time.time()
result = compute_sum(data)
end_time = time.time()
serial_time = end_time - start_time
print("Sequential Execution Time:", serial_time)

Sequential Execution Time: 7.104873657226562e-05


# **Part 3**

In [5]:
def chunk_data(data, num_chunks):
    chunk_size = len(data) // num_chunks
    return [data[i * chunk_size : (i + 1) * chunk_size] for i in range(num_chunks)]

In [6]:
def parallel_compute(data):
  num_processes = multiprocessing.cpu_count()
  chunks = chunk_data(data, num_processes)
  with multiprocessing.Pool(processes=num_processes) as pool:
    results = pool.map(compute_sum, chunks)
    return sum(results)

In [7]:
start_time = time.time()
parallel_result = parallel_compute(data)
end_time = time.time()
parallel_time = end_time - start_time
print("Parallel Execution Time:", parallel_time)

Parallel Execution Time: 1.7978496551513672


# **Part 4**

In [8]:
speedup = serial_time / parallel_time
print("Speedup:", speedup)

Speedup: 3.951873081750197e-05


In [9]:
num_processors = multiprocessing.cpu_count()
efficiency = speedup / num_processors
print("Efficiency:", efficiency)

Efficiency: 1.9759365408750985e-05



| Version | Execution Time (seconds) |
| :--- | :--- |
| **Sequential** | `7.104873657226562e-05` |
| **Parallel** | `1.7978496551513672` |


# **Part 5**

In [20]:
data = list(range(1, 10_000_000))

def compute_sum(data):
  total = 0
  for x in data:
    total += x * x
  return total

cProfile.run("compute_sum(data)")


         114 function calls (112 primitive calls) in 0.685 seconds

   Ordered by: standard name

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.659    0.659    0.659    0.659 4008752622.py:3(compute_sum)
        1    0.000    0.000    0.000    0.000 <frozen abc>:121(__subclasscheck__)
        1    0.000    0.000    0.000    0.000 <frozen importlib._bootstrap>:1390(_handle_fromlist)
        1    0.000    0.000    0.659    0.659 <string>:1(<module>)
        1    0.000    0.000    0.000    0.000 attrsettr.py:43(__getattr__)
        1    0.000    0.000    0.000    0.000 attrsettr.py:66(_get_attr_opt)
        8    0.000    0.000    0.000    0.000 enum.py:1128(__new__)
       21    0.000    0.000    0.000    0.000 enum.py:1543(_get_value)
        5    0.000    0.000    0.000    0.000 enum.py:1550(__or__)
        2    0.000    0.000    0.000    0.000 enum.py:1561(__and__)
        8    0.000    0.000    0.000    0.000 enum.py:720(__call__)
        1    

2. I observed from my profiler chart that my computer spent almost all of its energy running the compute_sum function. This function took 0.659 seconds out of the total 0.685 seconds of the entire program, meaning that this single math loop is taking up roughly 95% of my total project time and is the clear area where I should focus any efforts to make my code faster.

3. I observed that the specific performance bottleneck in this code is the standard Python for loop sitting inside that compute_sum function. This is because my computer is forced to look at all 10,000,000 numbers 1 by 1, manually check the type of each number, multiply it by itself, and add it to the total, creating a massive traffic jam of 10,000,000 individual tasks that slows my whole process down.

# **Part 6**

## **Option A Chosen**

In [16]:
def compute_sum_optimized(data):
  return sum(x*x for x in data)

# **Part 7**

In [23]:
# Original Data is retained since Option A is chosen
data = list(range(1, 10_000_000))
# Original Compute Sum
def compute_sum(data):
  total = 0
  for x in data:
    total += x * x
  return total

#Sequential Time
start_time = time.time()
result = compute_sum(data)
end_time = time.time()
serial_time = end_time - start_time

print("Sequential Execution Time (Original):", serial_time)

#Parallel Time
def parallel_compute(data):
  num_processes = multiprocessing.cpu_count()
  chunks = chunk_data(data, num_processes)
  with multiprocessing.Pool(processes=num_processes) as pool:
    results = pool.map(compute_sum, chunks)
    return sum(results)

start_time = time.time()
parallel_result = parallel_compute(data)
end_time = time.time()
parallel_time = end_time - start_time

print("Parallel Execution Time (Original):", parallel_time)

#Optimized version from moodle
def compute_sum_optimized(data):
  return sum(x*x for x in data)

#Sequential Time
start_time = time.time()
result = compute_sum_optimized(data)
end_time = time.time()
serial_time = end_time - start_time

print("Sequential Execution Time (Optimized):", serial_time)

#Parallel Time
def parallel_compute(data):
  num_processes = multiprocessing.cpu_count()
  chunks = chunk_data(data, num_processes)
  with multiprocessing.Pool(processes=num_processes) as pool:
    results = pool.map(compute_sum_optimized, chunks)
    return sum(results)

start_time = time.time()
parallel_result = parallel_compute(data)
end_time = time.time()
parallel_time = end_time - start_time
print("Parallel Execution Time (Optimized):", parallel_time)


Sequential Execution Time (Original): 0.864499568939209
Parallel Execution Time (Original): 1.896388053894043
Sequential Execution Time (Optimized): 0.8761556148529053
Parallel Execution Time (Optimized): 2.0778541564941406


**1. Which version was fastest?**

  I observed that the original sequential version was the fastest of all tested methods. It completed the task in 0.864 seconds, which beat both of my parallel attempts and the sequential optimized code.

**2. What was the speedup achieved?**

  I calculated that no speedup was achieved because my base sequential execution was already the fastest. Since the original sequential time was 0.864 seconds and the parallel optimized time was 2.078 seconds, running the code in parallel actually resulted in a slowdown factor of about 2.4, meaning the parallel run took more than twice as long as the single-threaded run.

**3. Was efficiency high or low? Why?**

  I observed that the efficiency of the parallel execution was very low. This low efficiency occurred because the math operation being performed was too lightweight for a computer to execute. The overhead time required for Python to spin up multiple CPU processes, split the list of 10,000,000 items, pickle the data to send across cores, and gather the sums back together ended up taking significantly longer than the actual calculation itself.

**4. What bottleneck did profiling reveal?**

  The profiling I conducted earlier revealed that the major performance bottleneck in this project was the raw Python interpreter itself handling the loop. Because Python is a dynamic language, it has to repeatedly check the data type of every single number in that massive list of 10,000,000 elements, run the multiplication, and handle the running total addition on every single iteration.

**5. How did optimization affect performance?**

  I observed that the optimization actually caused a slight decrease in performance. The original sequential code ran in 0.864 seconds, while the code using the optimized generator expression took 0.876 seconds. This happens because generators have a small amount of internal iteration overhead to yield numbers one at a time, and running that overhead 10,000,000 times in pure Python ended up being slightly slower than running a standard raw for loop.